# Исследование и разработка моделей классификации влияния экономических новостей

Тема курсовой работы: **«Разработка автоматической диалоговой системы на основе языковой модели для анализа экономических новостей»**.

Дисциплина: **«Машинное обучение в семантическом и сетевом анализе»**.
Студент: **Лашуков Т. Д.** (группа ПМ22-1).
Научный руководитель: доцент, к.э.н. **Макрушин С. В.**

---

## 1. Подготовка окружения и импорт библиотек

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Добавляем путь к скриптам исследования в PYTHONPATH
sys.path.append(str(Path("..").resolve() / "scripts"))

from economic_news_research.data import load_news_dataset, split_news_dataset
from economic_news_research.eda import build_eda_summary
from economic_news_research.modeling import train_baseline_model
from economic_news_research.embedding import SentenceTransformerEmbedder, train_embedding_classifier
from economic_news_research.transformer import HuggingFaceTinyTransformerTrainer, train_tiny_transformer_classifier

## 2. Загрузка данных и первичный разведочный анализ (EDA)

В качестве исходного набора данных используется датасет экономических новостей с разметкой влияния: `data/raw/news_impact.csv`.
Выведем основные характеристики датасета.

In [ ]:
dataset_path = Path("../../data/raw/news_impact.csv")
if not dataset_path.exists():
    # Попробуем альтернативный путь относительно research/notebooks
    dataset_path = Path("../data/raw/news_impact.csv")
    if not dataset_path.exists():
        dataset_path = Path("../../data/raw/economic_news.csv")

print(f"Используемый датасет: {dataset_path.resolve()}")
df = pd.read_csv(dataset_path)
print(f"Размерность датасета: {df.shape}")
df.head()

### Проверка пропущенных значений и типов данных

In [ ]:
df.info()

### Анализ распределения классов (целевой переменной `impact`)
Влияние новостей классифицируется на три категории:
- `negative` (негативное)
- `neutral` (нейтральное)
- `positive` (положительное)

In [ ]:
plt.figure(figsize=(8, 5))
sns.countplot(data=df, x='impact', hue='impact', palette='viridis', legend=False)
plt.title("Распределение классов влияния экономических новостей")
plt.xlabel("Класс влияния")
plt.ylabel("Количество новостей")
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

### Анализ длин текстов новостей

In [ ]:
df['text_len'] = df['text'].astype(str).str.len()
plt.figure(figsize=(10, 6))
sns.histplot(df['text_len'], bins=50, kde=True, color='skyblue')
plt.title("Распределение длин текстов новостей (в символах)")
plt.xlabel("Длина текста")
plt.ylabel("Частота")
plt.grid(linestyle='--', alpha=0.5)
plt.show()

print(df['text_len'].describe())

## 3. Разделение выборки на обучающую, валидационную и тестовую

В соответствии с методологией исследования, датасет разделяется в соотношении:
- **60%** — Обучающая выборка (train) для настройки весов моделей.
- **20%** — Валидационная выборка (validation) для отслеживания переобучения и подбора гиперпараметров.
- **20%** — Тестовая выборка (test) для итоговой оценки качества моделей.

Разделение выполняется с использованием стратификации по целевой переменной `impact`, чтобы сохранить пропорции классов во всех выборках.

In [ ]:
# Валидируем и загружаем датасет с нормализацией текстов
clean_df = load_news_dataset(dataset_path)
split = split_news_dataset(clean_df, random_state=42)

print(f"Train size: {len(split.train)} ({len(split.train)/len(clean_df):.1%})")
print(f"Validation size: {len(split.validation)} ({len(split.validation)/len(clean_df):.1%})")
print(f"Test size: {len(split.test)} ({len(split.test)/len(clean_df):.1%})")

## 4. Обучение и оценка моделей

Будет обучено три альтернативных архитектуры моделей:
1. **Baseline**: TF-IDF векторизация текста + Логистическая регрессия с подбором параметров.
2. **Embedding Classifier**: Векторные представления предложений (`SentenceTransformer`) + Логистическая регрессия.
3. **Deep Learning Model (PyTorch/HuggingFace)**: Дообучение легковесного трансформера BERT (`google/bert_uncased_L-2_H-128_A-2`).

Для сокращения времени выполнения экспериментов в ноутбуке можно ограничить число строк датасета (например, до 2000), используя стратифицированное сэмплирование.

In [ ]:
# Для ускорения обучения на локальной машине сэмплируем подвыборку
from economic_news_research.data import sample_news_dataset

sampled_df = sample_news_dataset(clean_df, max_rows=2000, random_state=42)
sampled_split = split_news_dataset(sampled_df, random_state=42)

print(f"Размер подвыборки для экспериментов: {len(sampled_df)}")

### 4.1. Baseline модель (TF-IDF + Логистическая регрессия)

Здесь выполняется автоматический подбор гиперпараметров (Grid Search):
- `tfidf__max_features`: размер словаря (20 000, 50 000)
- `tfidf__ngram_range`: униграммы и биграммы
- `classifier__C`: сила регуляризации логистической регрессии (1.0, 3.0)

In [ ]:
print("--- Обучение Baseline модели ---")
baseline_result = train_baseline_model(sampled_split, random_state=42)
print(f"Лучшие параметры: {baseline_result.best_params}")
print(f"F1-macro на валидации: {baseline_result.validation_metrics.f1_macro:.4f}")
print(f"F1-macro на тесте: {baseline_result.test_metrics.f1_macro:.4f}")

### 4.2. Модель на основе Эмбеддингов (SentenceTransformer + Логистическая регрессия)

Используется мультиязычная модель `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2` для извлечения семантических векторов текстов.

In [ ]:
print("--- Обучение модели на эмбеддингах ---")
# Используем кэширование эмбеддингов для ускорения
embedder = SentenceTransformerEmbedder()
embedding_result = train_embedding_classifier(sampled_split, embedder=embedder, random_state=42)
print(f"Лучшие параметры: {embedding_result.best_params}")
print(f"F1-macro на валидации: {embedding_result.validation_metrics.f1_macro:.4f}")
print(f"F1-macro на тесте: {embedding_result.test_metrics.f1_macro:.4f}")

### 4.3. Глубокая нейросетевая модель PyTorch (Tiny Transformer BERT)

Выполняется fine-tuning предобученной модели `google/bert_uncased_L-2_H-128_A-2` с добавлением полносвязного классификационного слоя на выходе.
Используется взвешенная функция CrossEntropyLoss для компенсации дисбаланса классов.

In [ ]:
print("--- Обучение PyTorch Tiny Transformer ---")
transformer_result = train_tiny_transformer_classifier(sampled_split, random_state=42)
print(f"F1-macro на валидации: {transformer_result.validation_metrics.f1_macro:.4f}")
print(f"F1-macro на тесте: {transformer_result.test_metrics.f1_macro:.4f}")

## 5. Сравнение и анализ результатов

Сведем полученные метрики качества и производительности моделей в общую таблицу для анализа.

In [ ]:
results_data = []
for r in [baseline_result, embedding_result, transformer_result]:
    results_data.append({
        "Модель": r.model_name,
        "Val F1-macro": r.validation_metrics.f1_macro,
        "Val Accuracy": r.validation_metrics.accuracy,
        "Test F1-macro": r.test_metrics.f1_macro,
        "Test Accuracy": r.test_metrics.accuracy,
        "Inference Time (sec/sample)": r.inference_seconds_per_sample
    })

results_df = pd.DataFrame(results_data)
results_df

### Визуализация сравнения F1-score моделей

In [ ]:
plt.figure(figsize=(10, 6))
sns.barplot(data=results_df, x='Модель', y='Test F1-macro', hue='Модель', palette='coolwarm', legend=False)
plt.title("Сравнение качества моделей по метрике Test F1-macro")
plt.ylabel("F1-macro")
plt.ylim(0, 1.0)
plt.grid(axis='y', linestyle='--', alpha=0.5)
for index, row in results_df.iterrows():
    plt.text(index, row['Test F1-macro'] + 0.02, f"{row['Test F1-macro']:.4f}", color='black', ha="center", fontweight='bold')
plt.show()

### Выводы

1. **Baseline (TF-IDF + LogReg)** показывает высокую скорость инференса и хорошее качество, являясь отличным решением при ограниченных вычислительных ресурсах.
2. **SentenceTransformer + LogReg** позволяет улавливать семантическую схожесть слов благодаря плотным эмбеддингам, однако модель инференса работает медленнее из-за этапа кодирования текста.
3. **Tiny Transformer (PyTorch BERT)** за счет механизма self-attention точнее учитывает контекстные связи в предложениях. При этом модель требует больше времени на обучение, но демонстрирует высокую обобщающую способность.

В диалоговой системе в качестве основной модели классификации влияния была выбрана и развернута **TF-IDF + LogisticRegression** из-за жестких требований к времени отклика в реальном времени на локальном стенде (CPU-инференс), тогда как тяжелые нейросети могут быть подключены при развертывании на серверах с GPU.